# Day 6: Advanced Analytics

This notebook computes Historical VaR, Rolling Sharpe Ratios, performs Investor Cohort Analysis, SIP Continuity tracking, and Sector HHI Concentration analysis.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import os
import warnings

warnings.filterwarnings('ignore')

os.makedirs('../reports/charts', exist_ok=True)
os.makedirs('../reports/data', exist_ok=True)

conn = sqlite3.connect('../bluestock_mf.db')

df_nav = pd.read_sql("SELECT * FROM fact_nav", conn)
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav = df_nav.sort_values(['amfi_code', 'date'])

df_fund = pd.read_sql("SELECT * FROM dim_fund", conn)

df_tx = pd.read_sql("SELECT * FROM fact_transactions", conn)
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

df_port = pd.read_sql("SELECT * FROM fact_portfolio", conn)
conn.close()


In [ ]:

# 1. Historical VaR (95%) and CVaR (95%)
df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()

def calc_var_cvar(group):
    rets = group['daily_return'].dropna()
    if len(rets) < 100:
        return pd.Series({'VaR_95': np.nan, 'CVaR_95': np.nan})
    var_95 = np.percentile(rets, 5)
    cvar_95 = rets[rets <= var_95].mean()
    return pd.Series({'VaR_95': var_95, 'CVaR_95': cvar_95})

df_risk = df_nav.groupby('amfi_code').apply(calc_var_cvar).reset_index()
df_risk = df_risk.merge(df_fund[['amfi_code', 'scheme_name']], on='amfi_code')
df_risk.to_csv('../reports/data/var_cvar_report.csv', index=False)
display(df_risk.sort_values('CVaR_95').head())


In [ ]:

# 2. Rolling 90-day Sharpe Ratio
rf_daily = 0.065 / 252
top_5 = df_fund['amfi_code'].head(5).tolist()

plt.figure(figsize=(14, 7))

for code in top_5:
    fund_data = df_nav[df_nav['amfi_code'] == code].set_index('date').sort_index()
    rets = fund_data['daily_return']
    
    roll_mean = rets.rolling(90).mean() - rf_daily
    roll_std = rets.rolling(90).std()
    roll_sharpe = (roll_mean / roll_std) * np.sqrt(252)
    
    name = df_fund[df_fund['amfi_code'] == code]['scheme_name'].values[0][:30]
    plt.plot(roll_sharpe.index, roll_sharpe.values, label=name, alpha=0.8)

plt.title('Rolling 90-Day Sharpe Ratio for Top 5 Funds')
plt.ylabel('Annualized Sharpe Ratio')
plt.xlabel('Date')
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/charts/rolling_sharpe_chart.png')
plt.show()


In [ ]:

# 3. Investor Cohort Analysis
# First transaction year
df_tx['year'] = df_tx['transaction_date'].dt.year
first_txn = df_tx.groupby('investor_id')['year'].min().reset_index().rename(columns={'year': 'cohort'})
df_cohort = df_tx.merge(first_txn, on='investor_id')

# Metrics per cohort
cohort_metrics = []
for cohort, group in df_cohort.groupby('cohort'):
    total_invested = group['amount_inr'].sum()
    sip_tx = group[group['transaction_type'] == 'Sip']
    avg_sip = sip_tx['amount_inr'].mean() if not sip_tx.empty else 0
    top_fund = group.groupby('amfi_code')['amount_inr'].sum().idxmax()
    top_fund_name = df_fund[df_fund['amfi_code'] == top_fund]['scheme_name'].values[0] if not pd.isna(top_fund) else "N/A"
    
    cohort_metrics.append({
        'Cohort Year': cohort,
        'Investors': group['investor_id'].nunique(),
        'Total Invested (INR)': total_invested,
        'Avg SIP Amount (INR)': avg_sip,
        'Top Fund Preference': top_fund_name
    })

df_cohort_analysis = pd.DataFrame(cohort_metrics)
display(df_cohort_analysis)


In [ ]:

# 4. SIP Continuity Analysis
sip_only = df_tx[df_tx['transaction_type'] == 'Sip'].sort_values(['investor_id', 'transaction_date'])
sip_counts = sip_only.groupby('investor_id').size()
valid_investors = sip_counts[sip_counts >= 6].index

sip_valid = sip_only[sip_only['investor_id'].isin(valid_investors)]
sip_valid['prev_date'] = sip_valid.groupby('investor_id')['transaction_date'].shift(1)
sip_valid['gap_days'] = (sip_valid['transaction_date'] - sip_valid['prev_date']).dt.days

avg_gap = sip_valid.groupby('investor_id')['gap_days'].mean().reset_index()
avg_gap['At_Risk'] = avg_gap['gap_days'] > 35

at_risk_count = avg_gap['At_Risk'].sum()
total_tracked = len(avg_gap)
print(f"SIP Continuity tracking {total_tracked} investors with 6+ SIPs.")
print(f"Identified {at_risk_count} 'At-Risk' investors (Avg gap > 35 days).")
display(avg_gap.head())


In [ ]:

# 5. Sector HHI Concentration
df_port['weight_pct_dec'] = df_port['weight_pct'] / 100
df_port['sq_weight'] = df_port['weight_pct_dec'] ** 2

hhi_fund = df_port.groupby('amfi_code')['sq_weight'].sum().reset_index().rename(columns={'sq_weight': 'HHI'})
hhi_fund = hhi_fund.merge(df_fund[['amfi_code', 'scheme_name', 'category']], on='amfi_code')

# Filter to equity/sectoral where HHI makes most sense
hhi_equity = hhi_fund[hhi_fund['category'].str.contains('Equity|Sectoral', na=False, case=False)]
display(hhi_equity.sort_values('HHI', ascending=False).head(10))



## Advanced Insights from Analysis

1. **Highest VaR/CVaR Exposure:** Sectoral and Small-Cap funds exhibit the deepest 5th percentile drawdowns, indicating that tail risk in these categories is substantially higher than standard deviation alone suggests.
2. **Rolling Sharpe Volatility:** The 90-day rolling Sharpe ratio chart reveals that even top-tier funds experienced periods of negative risk-adjusted returns during the 2024 market corrections, emphasizing that static trailing Sharpe ratios can mask short-term volatility.
3. **Cohort Loyalty:** Investors from earlier cohorts (e.g., 2022) exhibit higher average SIP amounts than newer cohorts, indicating successful step-up SIP behavior and compounding trust in the platform.
4. **SIP At-Risk Behavior:** A measurable fraction of seasoned investors (6+ SIPs) show an average gap of >35 days, flagging them as "At-Risk" for mandate failures or paused auto-pays, representing an immediate churn-prevention opportunity.
5. **HHI Concentration Risk:** Sectoral funds display HHI indices significantly closer to 1.0 (highly concentrated) compared to Multi-Cap funds, validating that active sector bets carry substantial idiosyncratic risk requiring close monitoring.


In [ ]:

# B3: Monte Carlo simulation projecting NAV growth over 5 years
top_fund = df_fund.iloc[0]['amfi_code'] # Use first fund
fund_data = df_nav[df_nav['amfi_code'] == top_fund].set_index('date')
returns = fund_data['daily_return'].dropna()
mu = returns.mean()
sigma = returns.std()

last_nav = fund_data['nav'].iloc[-1]
days = 252 * 5 # 5 years
simulations = 100

sims = np.zeros((days, simulations))
sims[0] = last_nav

np.random.seed(42)
for t in range(1, days):
    shock = np.random.normal(loc=mu, scale=sigma, size=simulations)
    sims[t] = sims[t-1] * (1 + shock)

plt.figure(figsize=(10, 6))
plt.plot(sims, color='blue', alpha=0.05)
plt.plot(np.percentile(sims, 5, axis=1), color='red', linestyle='--', label='5th Percentile (Worst Case)')
plt.plot(np.percentile(sims, 50, axis=1), color='green', linewidth=2, label='Median Expected')
plt.plot(np.percentile(sims, 95, axis=1), color='orange', linestyle='--', label='95th Percentile (Best Case)')
plt.title(f'Monte Carlo Simulation: 5-Year Projection for {top_fund}')
plt.xlabel('Trading Days')
plt.ylabel('Projected NAV')
plt.legend()
plt.savefig('../reports/charts/monte_carlo_sim.png')
plt.show()


In [ ]:

# B4: Markowitz Efficient Frontier
import scipy.optimize as sco

top_5_codes = top_5
df_pivot = df_nav[df_nav['amfi_code'].isin(top_5_codes)].pivot(index='date', columns='amfi_code', values='nav').dropna()
returns = df_pivot.pct_change().dropna()

mean_returns = returns.mean()
cov_matrix = returns.cov()
num_portfolios = 5000
risk_free_rate = 0.065 / 252

results = np.zeros((3, num_portfolios))
weights_record = []

for i in range(num_portfolios):
    weights = np.random.random(5)
    weights /= np.sum(weights)
    weights_record.append(weights)
    
    port_return = np.sum(mean_returns * weights) * 252
    port_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
    
    results[0,i] = port_std
    results[1,i] = port_return
    results[2,i] = (port_return - 0.065) / port_std

plt.figure(figsize=(10, 6))
plt.scatter(results[0,:], results[1,:], c=results[2,:], cmap='YlGnBu', marker='o', s=10, alpha=0.3)
plt.colorbar(label='Sharpe Ratio')
plt.title('Markowitz Efficient Frontier for Top 5 Funds')
plt.xlabel('Expected Volatility (Risk)')
plt.ylabel('Expected Return')

# Plot Max Sharpe
max_sharpe_idx = np.argmax(results[2])
plt.scatter(results[0,max_sharpe_idx], results[1,max_sharpe_idx], marker='*', color='r', s=200, label='Max Sharpe Portfolio')
plt.legend()
plt.savefig('../reports/charts/efficient_frontier.png')
plt.show()
